# Mean of Brand perception items
print("\nMean of Brand1:", round(df['Brand1'].mean(), 2))


In [ ]:

### Student Task
Create `StudentName_Descriptive.py` in Visual Studio Code. Calculate the mean and standard deviation specifically for the `Price1` column. Print both values. Observe the standard deviation. 

### Evaluation Questions
1. What is central tendency in statistics?
2. How does standard deviation explain the dispersion of the data?
3. Why do we need to know the frequencies of demographic variables?
4. What pandas method provides a quick statistical summary of a numerical column?
5. How can descriptive statistics help determine if a sample represents a target population?

## Section 3: Outlier Detection

### Objective
To use Mahalanobis Distance to identify multivariate outliers that may skew the analysis results.

**Keywords:** Mahalanobis distance, multivariate outlier, chi-square distribution, covariance matrix, p-value threshold, data quality

### Statistical Perspective
An outlier is a data point that differs significantly from other observations, often the result of data entry errors or truly anomalous respondent behavior. In univariate analysis, a boxplot handles this easily. In multivariate analysis, assumptions revolve around normality and the absence of extreme deviations across combined variables. A point might not be an outlier in any single variable but could be anomalous when predicting the relationship matrix.

#### Single-Variable Outlier Detection (IQR Method)

For a single variable, the standard approach uses the **Interquartile Range (IQR)**:

$$IQR = Q_3 - Q_1$$

where $Q_1$ is the 25th percentile and $Q_3$ is the 75th percentile of the distribution. A data point $x$ is flagged as an outlier if it falls outside the **inner fences**:

$$x < Q_1 - 1.5 \times IQR \quad \text{or} \quad x > Q_3 + 1.5 \times IQR$$

Values beyond $3 \times IQR$ from the quartiles are called **extreme outliers**. A **boxplot** directly visualises these boundaries: the box spans $Q_1$ to $Q_3$, the whiskers extend to the fences, and any points plotted beyond the whiskers are outliers.

### Python Example Code (Univariate)
```{python}
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('gogoro_data.csv')

fig, axes = plt.subplots(1, 3, figsize=(10, 5))
for ax, col in zip(axes, ['Func1', 'Price1', 'Brand1']):
    ax.boxplot(df[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_title(col)
    ax.set_ylabel('Score')

    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    ax.set_xlabel(f'Outliers: {n_out}')

plt.suptitle('Univariate Outlier Detection via Boxplot')
plt.tight_layout()
plt.show()



**Mahalanobis Distance ($D^2$):** A multidimensional measure calculating how far a data point is from the multi-dimensional center of all variables, adjusting for covariance (how variables correlate with each other, scaling elliptical structures).

**Hypothesis Test (Mahalanobis $\chi^2$):**

- **$H_0$ (Null Hypothesis):** The observation's multivariate distance from the centroid is consistent with the assumed normal distribution — it is not an outlier.
- **$H_1$ (Alternative Hypothesis):** The distance is significantly large, indicating a statistically anomalous data point (multivariate outlier).

Decision rule: If $D^2 > \chi^2_{k,\,\alpha}$, reject $H_0$ and flag the case as an outlier.

**Criteria & Assumptions:**

1. The structured data generally follows a multivariate normal distribution.
2. The critical value for identifying a multivariate outlier is determined using the $\chi^2$ distribution, with degrees of freedom equivalent to the number of predictor variables ($df = k$).
3. Commonly, cases with $p < 0.001$ on the $\chi^2$ distribution curve are deemed extreme multivariate outliers and should be removed. Failing to do so can heavily skew regression hyperplanes or distort factor topologies.

**Industry Application:** In anti-fraud divisions within banking, Mahalanobis Distance is aggressively implemented to detect anomalous credit card behaviors (where individual transaction amounts might be normal, but the combined velocity, location, and amounts deviate from the owner's multi-dimensional covariance matrix).

### Python Example Code


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import scipy.spatial.distance

df = pd.read_csv('gogoro_data.csv')
X = df[['Func1', 'Price1', 'Brand1']]

m_dist = [
    scipy.spatial.distance.mahalanobis(
        x, 
        X.mean(), 
        np.linalg.inv(X.cov()
        )
    ) for x in X.values
]

X = X.copy()
X['Mahalanobis'] = m_dist

crit_val = stats.chi2.ppf((1-0.01), df=3)
outliers = X[X['Mahalanobis'] > crit_val]
print("Number of outliers detected:", len(outliers))



> **Interpreting the Output:** The count represents respondents whose squared Mahalanobis distance exceeds the $\chi^2$ critical value at $\alpha = 0.01$ with $df = 3$. For each flagged case, $H_0$ is rejected — the respondent's combined pattern across Func1, Price1, and Brand1 is statistically anomalous relative to the multivariate distribution of the full sample. Removing these cases before structural analysis reduces the risk of distorted regression coefficients and inflated standard errors.

### Student Task
Create `StudentName_Outlier.py` in Visual Studio Code. Modify the alpha threshold of the $\chi^2$ test to $0.05$ instead of $0.01$ and observe the change in the number of outliers detected.

### Evaluation Questions
1. What is the difference between a univariate and a multivariate outlier?
2. What does Mahalanobis distance measure?
3. Why are multivariate outliers problematic in survey data analysis?
4. What role does the $\chi^2$ distribution play in outlier detection?
5. Statistically, what does the covariance matrix represent?

## Section 4: Normality Testing

### Objective
To assess the distribution of responses using Skewness and Kurtosis to ensure data fits the assumptions of standard regression.

**Keywords:** skewness, kurtosis, normal distribution, parametric assumption, bell curve, Shapiro-Wilk, data transformation

### Statistical Perspective
**Assumptions of Parametric Tests:** Many foundational statistical procedures (like standard OLS regression, ANOVA, or maximum likelihood SEM) demand that the residuals of the model or the underlying data distribution natively follow a normal Gaussian curve (a symmetric "bell curve").

**Skewness & Kurtosis Evaluation:**
Instead of solely relying on rigid visual histograms, statisticians use rigorous numerical metrics:

- *Skewness* measures the lateral lack of symmetry. For instance, if an overwhelmingly positive brand image causes most survey responses to cluster strongly around "5 - Strongly Agree", the data is negatively skewed. A general criterion is that skewness statistics should be strictly between -2 and +2.
- *Kurtosis* measures the absolute "tailedness" (the prevalence of extreme highs or lows producing heavy or light tails compared to a normal distribution curve). The conservative criterion for acceptable kurtosis is generally between -3 and +3 for robust SEM techniques.

**Hypothesis Test (Shapiro-Wilk):**

- **$H_0$ (Null Hypothesis):** The variable follows a normal distribution.
- **$H_1$ (Alternative Hypothesis):** The variable does not follow a normal distribution.

Decision rule: If $p < 0.05$, reject $H_0$ — the distribution significantly deviates from normality and non-parametric or bootstrapped methods are required.

**Consequences of Violations:** If the data significantly deviates from normality (i.e., fails strict normality tests like Shapiro-Wilk or Kolmogorov-Smirnov where $p < 0.05$), standard error estimations become completely unreliable. Non-parametric parametric procedural alternatives or specific techniques like bootstrapping (re-sampling the empirical data thousands of times programmatically to establish robust confidence interval standard errors independently of normal assumptions) must be fundamentally used.

**Industry Application:** In algorithmic stock trading, asset returns rarely follow a perfect normal distribution (often featuring "fat tails"/high kurtosis). Ignoring kurtosis leads to catastrophic underestimation of financial crash risks.

### Python Example Code


In [ ]:
import pandas as pd
from scipy import stats

df = pd.read_csv('gogoro_data.csv')

